In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# An Iterative Implementation of Merge Sort

The function `merge` takes five arguments.
  - `L`      is a list,
  - `start`  is an integer such that $\texttt{start}  \in \{0, \cdots, \texttt{len}(L)-1 \}$,
  - `middle` is an integer such that $\texttt{middle} \in \{0, \cdots, \texttt{len}(L)-1 \}$,
  - `end`    is an integer such that $\texttt{end}    \in \{0, \cdots, \texttt{len}(L)-1 \}$, 
  - `A`      is a list of the same length as `L`.
  
Furthermore, the indices `start`, `middle` and `end` have to satisfy the following inequations:
$$ 0 \leq \texttt{start} < \texttt{middle} < \texttt{end} \leq \texttt{len}(L) $$
The function assumes that the sublists `L[start:middle]` and `L[middle:end]` are already sorted.
The function merges these sublists so that when the call returns the sublist `L[start:end]`
is sorted.  The last argument `A` is used as auxiliary memory.

In [ ]:
function merge(L: number[], start: number, middle: number, end: number, A: number[]): void {
    for (let i = start; i < end; i++) {
        A[i] = L[i];
    }
    let idx1 = start;
    let idx2 = middle;
    let i = start;
    while (idx1 < middle && idx2 < end) {
        if (A[idx1] <= A[idx2]) {
            L[i] = A[idx1];
            idx1++;
        } else {
            L[i] = A[idx2];
            idx2++;
        }
        i++;
    }
    while (idx1 < middle) {
        L[i] = A[idx1];
        idx1++;
        i++;
    }
    while (idx2 < end) {
        L[i] = A[idx2];
        idx2++;
        i++;
    }
}

The function `mergeSort` is called with 2 arguments.
  - The first parameter $\texttt{L}$ is the list that is to be sorted.
  - The second parameter $\texttt{A}$ is used as an auxiliary array.  This array is needed
    as <em style="color:blue">temporary storage</em> and is required to have the same size as the list $\texttt{L}$.

The implementation uses two loops:
* The outer while loop sorts sublists of length `n`. Before the $\texttt{n}^{\mbox{th}}$ iteration of the outer while
  loop, all sublists of the form `L[n*k:n*(k+1)]` are sorted.  After the $\texttt{n}^{\mbox{th}}$ iteration, all
  sublists of the form `L[2*n*k:2*n*(k+1)]` are sorted.
* The inner while loop merges the sublists `L[n*k:n*(k+1)]` and `L[n*(k+1):n*(k+2)]` for even values of `k`.

In [ ]:
function mergeSort(L: number[], A: number[]): void {
    let n = 1;
    while (n < L.length) {
        let k = 0;
        while (n * k + n < L.length) {
            const top = Math.min(n * k + 2 * n, L.length);
            merge(L, n * k, n * k + n, top, A);
            k += 2;
        }
        n *= 2;
    }
}

The function $\texttt{sort}(L)$ sorts the list $L$ in place using <em style="color:blue">merge sort</em>.
It takes advantage of the fact that, in *Python*, lists are stored internally as arrays.
The function `sort` is a wrapper for the function `merge_sort`.  Its sole purpose is to allocate the auxiliary array `A`, 
which has the same size as the array storing `L`.

In [ ]:
function sort(L: number[]): void {
    const A = [...L];
    mergeSort(L, A);
}

## Testing

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

In [ ]:
function demo(): void {
    const L: number[] = Array.from({ length: 19 }, () => Math.floor(Math.random() * 99) + 1);
    console.log("L =", L);
    let S = [...L];
    sort(S);
    console.log("S =", S);
    const counterL = counter(L);
    const counterS = counter(S);
    console.log(counterL);
    console.log(counterS);
    console.log(compareCounters(counterL, counterS));
}


In [ ]:
demo();

The function `isOrdered(L)` checks that the list `L` is sorted in ascending order.

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `sameElements(L, S)` returns `true` if the lists `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function `randomIntRange(min, max)` generates a random integer in the range `[min, max)` and corresponds to Python's `range(min, max)` behavior regarding the exclusive upper bound.

In [ ]:
function randomIntRange(min: number, max: number): number {
  return Math.floor(Math.random() * (max - min)) + min;
} 

The function $\texttt{testSort}(n, k)$ generates $n$ random lists of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
    const oldL = [...L];
    sort(L);
    isOrdered(L);
    sameElements(oldL, L);
    process.stdout.write(".");
  }
  console.log("\nAll tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 20000);
console.timeEnd("testSort");

In [ ]:
console.time("1 million random integers");
const k = 1_000_000;
const L: number[] = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
sort(L);
console.timeEnd("1 million random integers");

*TypeScript* provides a built-in `sort()` method to sort an array of numbers.  
Let's see how it compares to our custom implementation.

In [ ]:
const k = 1_000_000;
const runs = 5;
for (let r = 0; r < runs; r++) {
    console.time("native sort");
    const L: number[] = Array.from({ length: k }, () => randomIntRange(0, 2 * k))
    const S = L.sort((a, b) => a - b);
    console.timeEnd("native sort");
}